In [0]:
# Databricks notebook source
from pyspark.sql.functions import col, to_date, trim

df_conditions_bronze = spark.read.table("workspace.bronze.conditions")

df_conditions_silver = (
    df_conditions_bronze
    .filter(col("PATIENT").isNotNull() & col("CODE").isNotNull())
    .select(
        to_date(col("START"), "yyyy-MM-dd").alias("onset_date"),
        to_date(col("STOP"), "yyyy-MM-dd").alias("resolved_date"),
        col("PATIENT").alias("patient_id"),
        col("ENCOUNTER").alias("encounter_id"),
        col("CODE").alias("snomed_code"),
        trim(col("DESCRIPTION")).alias("condition_description")
    )
)

(
    df_conditions_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.conditions")
)

print(f"✅ Created workspace.silver.conditions with {df_conditions_silver.count()} rows")